In [325]:
import os
import re
import pandas as pd
import itertools

In [326]:
item_map = {'0': 'Bird', '1': 'Drone'}
light_map = {'0': 'Clear', '1': 'Low', '2': 'Night'}
distance_map = {'0': '< 50m', '1': '> 50m'}
background_map = {'0': 'Clear', '1': 'Cluttered'}
drone_map = {'0': 'Small (Mavic 3 Pro)', '1': 'Big (Matrice 4T)', 'X': 'N/A'}
sensor_map = {'0': 'EO', '1': 'IR'}

In [327]:
dataset_path = r"/media/emre/SEB/DATASET" #just change this to what the path actually is

pattern = re.compile(
    r"(?P<Item>[01])_"
    r"(?P<LDBT>[012X]{4})_"
    r"(?P<Sensor>[01])_"
    r".+"
)

records = []

In [328]:
for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    match = pattern.fullmatch(folder)
    if not match:
        print(f"Skipping invalid folder: {folder}")
        continue

    bits = match.group("LDBT")

    light = bits[0]
    distance = bits[1]
    background = bits[2]
    drone_type = bits[3]

    frame_count = len([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.png'))
    ])

    records.append({
        "Item": item_map[match.group("Item")],
        "Light": light_map[light],
        "Distance": distance_map[distance],
        "Background": background_map[background],
        "Drone Type": drone_map[drone_type],
        "Sensor": sensor_map[match.group("Sensor")],
        "Frames": frame_count
    })

In [329]:
df = pd.DataFrame(records)
df # the drone type column is useless unless it server a purpose in your analysis

,Item,Light,Distance,Background,Drone Type,Sensor,Frames
0,Bird,Clear,> 50m,Clear,N/A,EO,36
1,Drone,Night,< 50m,Clear,Small (Mavic 3 Pro),EO,44
2,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,77
3,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,77
4,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,47
5,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,93
6,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,77
7,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,77
8,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,119
9,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,93


In [330]:
df.groupby(
    ["Item", "Light", "Distance", "Background", "Drone Type", "Sensor"]
)["Frames"].sum().reset_index()

,Item,Light,Distance,Background,Drone Type,Sensor,Frames
0,Bird,Clear,> 50m,Clear,N/A,EO,509
1,Bird,Clear,> 50m,Clear,N/A,IR,509
2,Drone,Clear,< 50m,Cluttered,Big (Matrice 4T),EO,911
3,Drone,Clear,< 50m,Cluttered,Big (Matrice 4T),IR,911
4,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,294
5,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,366
6,Drone,Clear,> 50m,Cluttered,Big (Matrice 4T),EO,1713
7,Drone,Clear,> 50m,Cluttered,Big (Matrice 4T),IR,1713
8,Drone,Clear,> 50m,Cluttered,Small (Mavic 3 Pro),EO,448
9,Drone,Clear,> 50m,Cluttered,Small (Mavic 3 Pro),IR,448


In [331]:
df.groupby("Sensor")["Frames"].sum() # we have an issue here, 72 frames difference

Sensor
EO    4984
IR    5056
Name: Frames, dtype: int64

In [332]:
df.groupby("Item")["Frames"].sum()

Item
Bird     1018
Drone    9022
Name: Frames, dtype: int64

## Finding the mismatch in EO and IR frames

In [333]:
df.groupby("Sensor")["Frames"].sum()

Sensor
EO    4984
IR    5056
Name: Frames, dtype: int64

In [334]:
df.groupby("Sensor").size()

Sensor
EO    81
IR    81
dtype: int64

In [335]:
df["Config"] = (
    df["Item"] + "_" +
    df["Light"] + "_" +
    df["Distance"] + "_" +
    df["Background"] + "_" +
    df["Drone Type"]
)

pairs = df.groupby("Config")["Sensor"].nunique()
pairs[pairs != 2]

Series([], Name: Sensor, dtype: int64)

In [336]:
pivot = df.pivot_table(
    index=["Item", "Light", "Distance", "Background", "Drone Type"],
    columns="Sensor",
    values="Frames",
    aggfunc="sum"
)

pivot[pivot["EO"] != pivot["IR"]]

,,,,Sensor,EO,IR
Item,Light,Distance,Background,Drone Type,,
Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),294,366


In [337]:
### Finding the specific folders where this happened

In [338]:
records = []

for folder in os.listdir(dataset_path):
    folder_path = os.path.join(dataset_path, folder)

    if not os.path.isdir(folder_path):
        continue

    match = pattern.fullmatch(folder)
    if not match:
        print(f"Skipping invalid folder: {folder}")
        continue

    bits = match.group("LDBT")

    frame_count = len([
        f for f in os.listdir(folder_path)
        if f.lower().endswith(('.jpg', '.png'))
    ])

    records.append({
        "Folder": folder,
        "Item": item_map[match.group("Item")],
        "Light": light_map[bits[0]],
        "Distance": distance_map[bits[1]],
        "Background": background_map[bits[2]],
        "Drone Type": drone_map[bits[3]],
        "Sensor": sensor_map[match.group("Sensor")],
        "Frames": frame_count
    })

df = pd.DataFrame(records)


In [339]:
subset = df[
    (df["Item"] == "Drone") &
    (df["Light"] == "Clear") &
    (df["Distance"] == "< 50m") &
    (df["Background"] == "Cluttered") &
    (df["Drone Type"] == "Small (Mavic 3 Pro)")
]

subset[["Sensor", "Folder", "Frames"]]


,Sensor,Folder,Frames
2,EO,1_0010_0_E19,77
3,EO,1_0010_0_E20,77
4,EO,1_0010_0_E21,47
5,EO,1_0010_0_E22,93
6,IR,1_0010_1_E19,77
7,IR,1_0010_1_E20,77
8,IR,1_0010_1_E21,119
9,IR,1_0010_1_E22,93


In [340]:
mismatches = pivot[pivot["EO"] != pivot["IR"]].reset_index()

for _, row in mismatches.iterrows():
    subset = df[
        (df["Item"] == row["Item"]) &
        (df["Light"] == row["Light"]) &
        (df["Distance"] == row["Distance"]) &
        (df["Background"] == row["Background"]) &
        (df["Drone Type"] == row["Drone Type"])
    ]
    print(subset[["Sensor", "Folder", "Frames"]])


  Sensor        Folder  Frames
2     EO  1_0010_0_E19      77
3     EO  1_0010_0_E20      77
4     EO  1_0010_0_E21      47
5     EO  1_0010_0_E22      93
6     IR  1_0010_1_E19      77
7     IR  1_0010_1_E20      77
8     IR  1_0010_1_E21     119
9     IR  1_0010_1_E22      93


In [341]:
# for now, i will just remove these folders and build the frequency table without them

In [342]:
exclude_folders = {
    "1_0010_1_E21",
    "1_0010_0_E21"
}

df_filtered = df[~df["Folder"].isin(exclude_folders)].copy()

In [343]:
pivot_filtered = df_filtered.pivot_table(
    index=["Item", "Light", "Distance", "Background", "Drone Type"],
    columns="Sensor",
    values="Frames",
    aggfunc="sum"
)

pivot_filtered[pivot_filtered["EO"] != pivot_filtered["IR"]]

,,,,Sensor,EO,IR
Item,Light,Distance,Background,Drone Type,,


## Continuing table

In [344]:
frequency_table = (
    df_filtered
    .groupby(
        ["Item", "Light", "Distance", "Background", "Drone Type", "Sensor"],
        as_index=False
    )["Frames"]
    .sum()
    .rename(columns={"Frames": "Total Frames"})
)

frequency_table


,Item,Light,Distance,Background,Drone Type,Sensor,Total Frames
0,Bird,Clear,> 50m,Clear,N/A,EO,509
1,Bird,Clear,> 50m,Clear,N/A,IR,509
2,Drone,Clear,< 50m,Cluttered,Big (Matrice 4T),EO,911
3,Drone,Clear,< 50m,Cluttered,Big (Matrice 4T),IR,911
4,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,247
5,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,247
6,Drone,Clear,> 50m,Cluttered,Big (Matrice 4T),EO,1713
7,Drone,Clear,> 50m,Cluttered,Big (Matrice 4T),IR,1713
8,Drone,Clear,> 50m,Cluttered,Small (Mavic 3 Pro),EO,448
9,Drone,Clear,> 50m,Cluttered,Small (Mavic 3 Pro),IR,448


In [345]:
frequency_table2 = (
    df_filtered
    .groupby(
        ["Item", "Light", "Distance", "Background", "Drone Type"],
        as_index=False
    )["Frames"]
    .sum()
    .rename(columns={"Frames": "Total Frames"})
)

frequency_table2


,Item,Light,Distance,Background,Drone Type,Total Frames
0,Bird,Clear,> 50m,Clear,N/A,1018
1,Drone,Clear,< 50m,Cluttered,Big (Matrice 4T),1822
2,Drone,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),494
3,Drone,Clear,> 50m,Cluttered,Big (Matrice 4T),3426
4,Drone,Clear,> 50m,Cluttered,Small (Mavic 3 Pro),896
5,Drone,Night,< 50m,Clear,Small (Mavic 3 Pro),350
6,Drone,Night,> 50m,Clear,Small (Mavic 3 Pro),1868


In [346]:
items = ["Bird", "Drone"]
lights = ["Clear", "Low", "Night"]
distances = ["< 50m", "> 50m"]
backgrounds = ["Clear", "Cluttered"]
drone_types = ["Small (Mavic 3 Pro)", "Big (Matrice 4T)"]
sensors = ["EO", "IR"]

all_combinations = pd.DataFrame(list(itertools.product(
    items, lights, distances, backgrounds, drone_types, sensors
)), columns=["Item", "Light", "Distance", "Background", "Drone Type", "Sensor"])

agg = df_filtered.groupby(
    ["Item", "Light", "Distance", "Background", "Drone Type", "Sensor"]
)["Frames"].sum().reset_index()

full_table = all_combinations.merge(
    agg,
    on=["Item", "Light", "Distance", "Background", "Drone Type", "Sensor"],
    how="left"
)

full_table["Frames"] = full_table["Frames"].fillna(0).astype(int)

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

full_table

,Item,Light,Distance,Background,Drone Type,Sensor,Frames
0,Bird,Clear,< 50m,Clear,Small (Mavic 3 Pro),EO,0
1,Bird,Clear,< 50m,Clear,Small (Mavic 3 Pro),IR,0
2,Bird,Clear,< 50m,Clear,Big (Matrice 4T),EO,0
3,Bird,Clear,< 50m,Clear,Big (Matrice 4T),IR,0
4,Bird,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),EO,0
5,Bird,Clear,< 50m,Cluttered,Small (Mavic 3 Pro),IR,0
6,Bird,Clear,< 50m,Cluttered,Big (Matrice 4T),EO,0
7,Bird,Clear,< 50m,Cluttered,Big (Matrice 4T),IR,0
8,Bird,Clear,> 50m,Clear,Small (Mavic 3 Pro),EO,0
9,Bird,Clear,> 50m,Clear,Small (Mavic 3 Pro),IR,0


In [347]:
frames_by_item = df_filtered.groupby("Item")["Frames"].sum().reset_index()
frames_by_item

,Item,Frames
0,Bird,1018
1,Drone,8856


In [348]:
frames_by_light = (
    df_filtered.groupby("Light")["Frames"].sum()
    .reindex(lights, fill_value=0)
    .reset_index()
)
frames_by_light


,Light,Frames
0,Clear,7656
1,Low,0
2,Night,2218


In [349]:
frames_by_distance = df_filtered.groupby("Distance")["Frames"].sum().reset_index()
frames_by_distance

,Distance,Frames
0,< 50m,2666
1,> 50m,7208


In [350]:
frames_by_background = df_filtered.groupby("Background")["Frames"].sum().reset_index()
frames_by_background

,Background,Frames
0,Clear,3236
1,Cluttered,6638


In [351]:
frames_by_drone = df_filtered.groupby("Drone Type")["Frames"].sum().reset_index()
frames_by_drone

,Drone Type,Frames
0,Big (Matrice 4T),5248
1,N/A,1018
2,Small (Mavic 3 Pro),3608


In [352]:
frames_by_sensor = df_filtered.groupby("Sensor")["Frames"].sum().reset_index()
frames_by_sensor

,Sensor,Frames
0,EO,4937
1,IR,4937


grouped_tables = {
    "By_Item": ["Item"],
    "By_Light": ["Light"],
    "By_Distance": ["Distance"],
    "By_Background": ["Background"],
    "By_Drone_Type": ["Drone Type"],
    "By_Sensor": ["Sensor"],
}

with pd.ExcelWriter("all_grouped_tables.xlsx") as writer:
    for sheet_name, cols in grouped_tables.items():
        df_filtered.groupby(cols)["Frames"].sum().reset_index().to_excel(writer, sheet_name=sheet_name, index=False)

full_table.to_csv("full_table.csv", index=False)
full_table.to_excel("full_table.xlsx", index=False)

frequency_table.to_csv("frequency_table.csv", index=False)
frequency_table.to_excel("frequency_table.xlsx", index=False)

with pd.ExcelWriter("dataset_summary.xlsx") as writer:
    full_table.to_excel(writer, sheet_name="Full_Table", index=False)
    frequency_table.to_excel(writer, sheet_name="What_we_have", index=False)
    
    # Grouped summaries
    frames_by_item.to_excel(writer, sheet_name="By_Item", index=False)
    frames_by_light.to_excel(writer, sheet_name="By_Light", index=False)
    frames_by_distance.to_excel(writer, sheet_name="By_Distance", index=False)
    frames_by_background.to_excel(writer, sheet_name="By_Background", index=False)
    frames_by_drone.to_excel(writer, sheet_name="By_Drone_Type", index=False)
    frames_by_sensor.to_excel(writer, sheet_name="By_Sensor", index=False)

In [353]:
def group_frames(df, group_cols, factor_levels=None):
    if factor_levels is None:
        # simple groupby if no factor_levels provided
        return df.groupby(group_cols)["Frames"].sum().reset_index()
    
    # generate all possible combinations of levels
    all_combos = pd.DataFrame(list(itertools.product(
        *(factor_levels[col] for col in group_cols)
    )), columns=group_cols)
    
    # aggregate actual frames
    agg = df.groupby(group_cols)["Frames"].sum().reset_index()
    
    # merge and fill missing
    full_grouped = all_combos.merge(agg, on=group_cols, how="left")
    full_grouped["Frames"] = full_grouped["Frames"].fillna(0).astype(int)
    
    return full_grouped


In [354]:
factor_levels = {
    "Item": ["Bird", "Drone"],
    "Light": ["Clear", "Low", "Night"],
    "Distance": ["< 50m", "> 50m"],
    "Background": ["Clear", "Cluttered"],
    "Drone Type": ["Small (Mavic 3 Pro)", "Big (Matrice 4T)"],
    "Sensor": ["EO", "IR"]
}

grouped_definitions = {
    "By_Item": ["Item"],
    "By_Light": ["Light"],
    "By_Distance": ["Distance"],
    "By_Background": ["Background"],
    "By_Drone_Type": ["Drone Type"],
    "By_Sensor": ["Sensor"],
}

In [355]:
all_grouped_tables = {}

for name, cols in grouped_definitions.items():
    levels = {col: factor_levels[col] for col in cols}
    all_grouped_tables[name] = group_frames(df_filtered, cols, levels)


In [356]:
all_grouped_tables["By_Light"]

,Light,Frames
0,Clear,7656
1,Low,0
2,Night,2218


In [357]:
with pd.ExcelWriter("all_grouped_tables.xlsx") as writer:
    full_table.to_excel(writer, sheet_name="Full_Table", index=False)
    frequency_table.to_excel(writer, sheet_name="What_we_have", index=False)
    for sheet_name, table in all_grouped_tables.items():
        table.to_excel(writer, sheet_name=sheet_name, index=False)